# 03 — Enrich building heights with EUBUCCO

The baseline VoxCity model contains many default building heights. This notebook retrieves EUBUCCO building records for the PoC domain, cleans the geometries, converts them to WGS84 for VoxCity, and updates the building-height grid.

This clean version consolidates the two exploratory building-height notebooks into one reproducible stage.

In [ ]:
from pathlib import Path

# The notebooks_v02 folder is expected to sit directly inside the repository.
# If you open Jupyter from another working directory, set PROJECT_ROOT manually.
HERE = Path.cwd().resolve()
PROJECT_ROOT = HERE.parent if HERE.name == "notebooks_v02" else HERE

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
OUTPUTS = PROJECT_ROOT / "outputs"

for folder in [DATA_RAW, DATA_INTERIM, OUTPUTS]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
import json
import duckdb
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from shapely import wkt
from voxcity.generator import get_voxcity, save_voxcity, update_voxcity

boundary = gpd.read_file(DATA_RAW / "sevilla_boundary.gpkg")
baseline = xr.open_dataset(DATA_INTERIM / "sevilla_voxcity_initial.nc")

## 1. Recreate the PoC footprint in ETRS89-LAEA (EPSG:3035)

In [ ]:
boundary_projected = boundary.to_crs(boundary.estimate_utm_crs())
centroid = boundary_projected.geometry.centroid.iloc[0]

poc_projected = gpd.GeoDataFrame(
    geometry=[centroid.buffer(1000).envelope],
    crs=boundary_projected.crs,
)

poc_3035 = poc_projected.to_crs(3035)
xmin, ymin, xmax, ymax = poc_3035.total_bounds

print("EUBUCCO query bounds:", xmin, ymin, xmax, ymax)

## 2. Query EUBUCCO

In [ ]:
con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute("INSTALL spatial;")
con.execute("LOAD spatial;")

con.execute("SET s3_endpoint='s3.eubucco.com';")
con.execute("SET s3_url_style='path';")
con.execute("SET s3_region='eu';")
con.execute("SET s3_access_key_id='';")
con.execute("SET s3_secret_access_key='';")

s3_path = "s3://eubucco/v0.2/buildings/parquet/*/*.parquet"

query = f"""
SELECT
    id,
    region_id,
    city_id,
    height,
    floors,
    type,
    subtype,
    height_source,
    geometry_source,
    ST_AsText(geometry) AS geometry_wkt
FROM read_parquet(
    '{s3_path}',
    hive_partitioning = true
)
WHERE
    region_id LIKE 'ES%'
    AND bbox.xmin <= {xmax}
    AND bbox.xmax >= {xmin}
    AND bbox.ymin <= {ymax}
    AND bbox.ymax >= {ymin}
"""

df_eubucco = con.execute(query).df()
print("Candidate EUBUCCO rows:", len(df_eubucco))

## 3. Convert, clip, and clean EUBUCCO geometries

In [ ]:
eubucco = gpd.GeoDataFrame(
    df_eubucco.drop(columns="geometry_wkt"),
    geometry=df_eubucco["geometry_wkt"].apply(wkt.loads),
    crs="EPSG:3035",
)

eubucco = gpd.clip(eubucco, poc_3035)

eubucco = eubucco[
    eubucco["height"].notna()
    & eubucco.geometry.notna()
    & eubucco.geometry.is_valid
].copy()

print("Clean EUBUCCO buildings:", len(eubucco))
display(eubucco["height"].describe())
display(eubucco["height_source"].value_counts(dropna=False))

In [ ]:
eubucco_clean = eubucco.to_crs(4326).copy()

# VoxCity requires simple identifiers and a minimum-height field.
eubucco_clean["id"] = range(1, len(eubucco_clean) + 1)
eubucco_clean["min_height"] = 0.0

eubucco_clean.to_file(
    DATA_INTERIM / "sevilla_eubucco_buildings.gpkg",
    driver="GPKG",
)

## 4. Update the VoxCity building-height layer

In [ ]:
rectangle_vertices = json.loads(
    baseline.attrs["rectangle_vertices"]
)

voxcity_model = get_voxcity(
    rectangle_vertices,
    meshsize=5,
)

voxcity_eubucco = update_voxcity(
    voxcity_model,
    building_gdf=eubucco_clean,
    inplace=False,
)

old_heights = baseline["building_height"].values
new_heights = voxcity_eubucco.buildings.heights

print("Enriched building cells:", int((new_heights > 0).sum()))
print("Mean enriched height:", float(new_heights[new_heights > 0].mean()))
print("Maximum enriched height:", float(new_heights.max()))

## 5. Compare baseline and enriched heights

In [ ]:
mask = (old_heights > 0) | (new_heights > 0)
difference = new_heights - old_heights

print("Mean absolute difference:", float(np.abs(difference[mask]).mean()))
print("Median absolute difference:", float(np.median(np.abs(difference[mask]))))
print("Maximum absolute difference:", float(np.abs(difference[mask]).max()))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(old_heights, origin="lower")
axes[0].set_title("Baseline heights")

axes[1].imshow(new_heights, origin="lower")
axes[1].set_title("EUBUCCO-enriched heights")

image = axes[2].imshow(difference, origin="lower")
axes[2].set_title("Height difference")

for ax in axes:
    ax.set_axis_off()

plt.colorbar(image, ax=axes[2], label="Difference (m)")
plt.tight_layout()
plt.show()

## 6. Save the enriched model

In [ ]:
save_voxcity(
    DATA_INTERIM / "sevilla_voxcity_eubucco.pkl",
    voxcity_eubucco,
)

enriched_ds = voxcity_eubucco.to_xarray()
enriched_to_save = enriched_ds.copy()
enriched_to_save.attrs["rectangle_vertices"] = json.dumps(
    np.asarray(enriched_ds.attrs["rectangle_vertices"]).tolist()
)

enriched_to_save.to_netcdf(
    DATA_INTERIM / "sevilla_voxcity_eubucco.nc"
)

print("Saved enriched VoxCity model.")